In [ ]:
# data i/o
import os
import subprocess
import zipfile
# for plots
import matplotlib.pyplot as plt
# the usual
import numpy as np
import pandas as pd
#import deepgraph as dg
# notebook display
%matplotlib inline
pd.options.display.max_rows = 10
pd.set_option('expand_frame_repr', False)

In [ ]:
# zip file containing node attributes
os.makedirs("tmp", exist_ok=True)
get_nodes_zip = ("wget -O tmp/terrorist_nodes.zip "
                 "https://sites.google.com/site/sfeverton18/"
                 "research/appendix-1/Noordin%20Subset%20%28ORA%29.zip?"
                 "attredirects=0&d=1")
import urllib.request
os.makedirs("tmp", exist_ok=True)
urllib.request.urlretrieve(get_nodes_zip.split()[-1], "tmp/terrorist_nodes.zip")
# unzip
zf = zipfile.ZipFile('tmp/terrorist_nodes.zip')
zf.extract('Attributes.csv', path='tmp/')
zf.close()
# create node table
v = pd.read_csv('tmp/Attributes.csv')
v.rename(columns={'Unnamed: 0': 'Name'}, inplace=True)
# create a copy of all nodes for each layer (i.e., create "node-layers")
# there are 10 layers and 79 nodes on each layer
v = pd.concat(10*[v])
# add "aspect" as column to v
layer_names = ['Business', 'Communication', 'O Logistics', 'O Meetings',
               'O Operations', 'O Training', 'T Classmates', 'T Friendship',
               'T Kinship', 'T Soulmates']
layers = [[name]*79 for name in layer_names]
layers = [item for sublist in layers for item in sublist]
v['layer'] = layers
# set unique node index
v.reset_index(inplace=True)
v.rename(columns={'index': 'V_N'}, inplace=True)
# swap columns
cols = list(v)
cols[1], cols[10] = cols[10], cols[1]
v = v[cols]
# get rid of the attribute columns for demonstrational purposes,
# will be inserted again later
v, vinfo = v.iloc[:, :2], v.iloc[:, 2:]


In [ ]:
import pandas as pd
import networkx as nx
#import openpyxl
import os
# -------------------------------------------------
# 1. Load node attributes
# -------------------------------------------------
nodes = pd.read_csv(r"D:\My_Doc\graph_know\NOORDIN\Noordin Top Terrorist Network Data.csv")
# -------------------------------------------------
# 2. Define layers (Noordin dataset의 10개 레이어)
# -------------------------------------------------
layer_names = [
    'Business', 'Communication', 'O Logistics', 'O Meetings',
    'O Operations', 'O Training', 'T Classmates', 'T Friendship',
    'T Kinship', 'T Soulmates'
]
# -------------------------------------------------
# 3. Create node-layer table (기존 코드의 concat(10*[v]) 대체)
# -------------------------------------------------
node_layers = []
for layer in layer_names:
    temp = nodes.copy()
    temp["layer"] = layer
    node_layers.append(temp)
v = pd.concat(node_layers, ignore_index=True)
# unique node-layer id
v.reset_index(inplace=True)
v.rename(columns={"index": "V_N"}, inplace=True)
# -------------------------------------------------
# 4. Load edges per layer
# -------------------------------------------------
graphs = {}
for layer in layer_names:
    layer_key = layer.lower().replace(" ", "_")
    edge_path = f"data/edges_{layer_key}.csv"
    if not os.path.exists(edge_path):
        continue
    e = pd.read_csv(edge_path)
    G = nx.Graph(layer=layer)
    for _, row in v[v["layer"] == layer].iterrows():
        G.add_node(
            row["node_id"],
            name=row["name"],
            role=row["role"],
            nationality=row["nationality"],
            organization=row["organization"],
            layer=layer
        )
    for _, row in e.iterrows():
        G.add_edge(row["src"], row["dst"], weight=row.get("weight", 1))
    graphs[layer] = G

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, seed=42)
nx.draw(
    G,
    pos,
    node_size=50,
    node_color="lightgray",
    edge_color="gray",
    alpha=0.7,
    with_labels=False
)
plt.title("Network Visualization (Spring Layout)")
plt.axis("off")
plt.show()